# 01 — Chamada à API e resposta em JSON

**Trilha:** **01 chamada e JSON** → [02 saída estruturada](02_saida_estruturada.ipynb) → [03 enums e guardrails](03_enums_e_guardrails.ipynb)
→ [intermediário](../02_intermediario/04_ferramentas_com_funcoes.ipynb) → [avançado](../03_avancado/07_contrato_e_contexto.ipynb)

O exemplo que acompanha a trilha inteira: **uma lista de compras gerada a partir de um prato**. Aqui ele começa como no seu
primeiro contato com a API: pedir JSON no prompt e ler com `json.loads`.

> **Como rodar.** As células marcadas com `# api:` chamam a API de verdade e **não foram executadas** neste repositório (não há
> chave aqui). Com o `.env` preenchido (veja [`.env.example`](../.env.example)), elas rodam normalmente. As demais células usam uma
> **resposta real gravada** ([`respostas_gravadas/`](../respostas_gravadas/README.md)) ou **exemplos construídos**, sempre rotulados.

## 1. Configuração (o mesmo molde do `AI_Agents_Basics`)

> **Saída colorida.** Resultados com cores pelo [`prettifier.py`](../prettifier.py) (baseado em
> [`rich`](https://github.com/Textualize/rich)) e eventos de execução pelo [`logger_general.py`](../logger_general.py).
> `show(...)` é o `print` com cores; `NO_COLOR=1` no ambiente desliga as cores.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # prettifier.py e logger_general.py ficam em AI_use_cases/
from prettifier import show, show_table

In [ ]:
# api: configuração do cliente — lê a chave e o endpoint do .env
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("openai")
if not api_key:
    raise ValueError("variável 'openai' (chave da API) não encontrada no .env")

openai_base_url = os.getenv("openai_base_url")
if not openai_base_url:
    raise ValueError("variável 'openai_base_url' não encontrada no .env")

AI_MODEL = "gpt-4o-mini"
ai_client = OpenAI(api_key=api_key, base_url=openai_base_url)

## 2. Olá, mundo

In [ ]:
# api: primeira chamada
chat_completion = ai_client.chat.completions.create(
    model=AI_MODEL,
    messages=[{"role": "user", "content": "Pode me dar algumas saudações, por favor?"}],
)
show(f"chat_completion.choices[0].message.content: {chat_completion.choices[0].message.content}")

A resposta vem em `choices[0].message.content`: **texto livre**. Para um programa usar a resposta, o texto precisa ter uma
forma previsível — o próximo passo.

## 3. Lista de compras em JSON, pedindo no prompt

In [ ]:
# api: pedindo JSON no prompt
SYSTEM_PROMPT = """
You are a grocery list generator. Given a meal as input, you return a list of ingredients.
The meal shall use whole food ingredients as much as possible.
You include the amount of each ingredient to prepare the meal for two persons.
You must use metric system in your answer.
Answer in JSON format: [{"ingredients": "...", "amount": "..."}]
"""

chat_completion = ai_client.chat.completions.create(
    model=AI_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Pizza margherita"},
    ],
)
resposta_ao_vivo = chat_completion.choices[0].message.content
show(f"resposta_ao_vivo: {resposta_ao_vivo}")

Daqui em diante usamos a **resposta real gravada** dessa mesma chamada (rodada pelo autor com `gpt-4o-mini`). Para usar a sua,
troque a linha abaixo por `resposta = resposta_ao_vivo`.

In [2]:
import json
from pathlib import Path

resposta = Path("../respostas_gravadas/pizza_margherita_real.json").read_text(encoding="utf-8")
show(f"len(resposta): {len(resposta)}")
show(f"resposta[:80]: {resposta[:80]}...")

len(resposta): 484
resposta[:80]: [{"ingredients": "All-purpose flour","amount": "250 grams"},{"ingredients": "Act...


## 4. Lendo o JSON

In [3]:
itens = json.loads(resposta)
show(f"type(itens).__name__: {type(itens).__name__}")
show(f"len(itens): {len(itens)}")
show_table(["ingredients", "amount"], [(item["ingredients"], repr(item["amount"])) for item in itens],
       title="resposta real, item a item")

type(itens).__name__: list
len(itens): 9
resposta real, item a item               
                                         
  ingredients               amount       
 ─────────────────────────────────────── 
  All-purpose flour         '250 grams'  
  Active dry yeast          '5 grams'    
  Salt                      '5 grams'    
  Water                     '150 ml'     
  Olive oil                 '15 ml'      
  Tomato sauce              '100 grams'  
  Fresh mozzarella cheese   '150 grams'  
  Fresh basil leaves        '10 grams'   
  Oregano (dried)           '1 gram'     
                                         


Funcionou: uma lista de dicionários. Mas o que exatamente o programa recebeu?

In [4]:
primeiro = itens[0]
show(f"primeiro: {primeiro}")
show(f"type(primeiro['amount']).__name__: {type(primeiro['amount']).__name__}")

primeiro: {'ingredients': 'All-purpose flour', 'amount': '250 grams'}
type(primeiro['amount']).__name__: str


## ⚠️ O que ainda quebra (e o [02](02_saida_estruturada.ipynb) resolve)

**1. A quantidade é texto.** `"250 grams"` mistura número e unidade. Para fazer a lista para **4 pessoas**, o programa precisaria
"adivinhar" onde está o número:

In [5]:
show(f"primeiro['amount'] * 2: {primeiro['amount'] * 2!r}")

primeiro['amount'] * 2: '250 grams250 grams'


Não deu erro — e esse é o problema: `"250 grams" * 2` é `"250 grams250 grams"`. Qualquer cálculo exige *parsing* manual, e o
modelo pode escrever `"250g"`, `"0.25 kg"` ou `"2 xícaras"`.

**2. O formato não é garantido.** O prompt **pede** JSON; nada obriga o modelo a obedecer. Duas respostas que aparecem na
prática (*exemplos construídos*):

In [6]:
com_cerca = 'Claro! Aqui está a lista:\n```json\n[{"ingredients": "Flour", "amount": "250 grams"}]\n```'
chave_trocada = '[{"ingredient": "Flour", "quantity": "250 grams"}]'

try:
    json.loads(com_cerca)
except json.JSONDecodeError as ex:
    show(f"com texto em volta: JSONDecodeError: {ex}")

lista_trocada = json.loads(chave_trocada)
try:
    show(f'lista_trocada[0]["ingredients"]: {lista_trocada[0]["ingredients"]}')
except KeyError as ex:
    show(f"com chave trocada: KeyError: {ex}")

com texto em volta: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
com chave trocada: KeyError: 'ingredients'


O primeiro quebra no `json.loads`; o segundo passa no `json.loads` e quebra **depois**, longe da chamada, quando alguém usa a chave.

**Resumo:** pedir JSON no prompt dá um texto que *parece* dado. Falta um **contrato**: quais campos, de que tipos — e um jeito de
conferir. ➡️ [02 — saída estruturada](02_saida_estruturada.ipynb)